# 📜 Historia API - Od SOAP do OpenAPI

**Kompletny przewodnik po standardach i filozofiach budowania API**

---

## 🎯 Czego się nauczysz?

1. **Czym jest API** - podstawy
2. **Historia chronologiczna** - RPC → SOAP → REST → GraphQL → gRPC
3. **SOAP** - standard, WSDL, samodokumentacja
4. **REST** - filozofia (nie standard!), zasady, ograniczenia
5. **RESTful** - best practices, różnice vs REST
6. **HATEOAS** - hipermedia, samodokumentacja przez linki
7. **OpenAPI** - specyfikacja (Swagger), aktualna wersja
8. **GraphQL** - alternatywa dla REST
9. **gRPC** - następca RPC, Google
10. **Porównanie wszystkich** - kiedy co używać

---

## 1. Czym jest API?

**API (Application Programming Interface)** = sposób komunikacji między aplikacjami

### Analogia: Restauracja

```
Ty (Klient)          Menu (API)          Kuchnia (Serwer)
     ↓                   ↓                      ↓
Wybierasz danie    Pokazuje opcje      Przygotowuje jedzenie
     ↓                   ↓                      ↓
Mówisz kelnerowi   Tłumaczy request    Wykonuje request
     ↓                   ↓                      ↓
Dostajesz jedzenie  Zwraca response    Wysyła gotowe danie
```

**API = kontrakt między aplikacjami:**
- "Jeśli wyślesz taki request, dostaniesz taki response"
- Nie musisz wiedzieć JAK to działa (implementacja)
- Musisz tylko wiedzieć CO zrobić (interfejs)

---

## 2. Historia API - Timeline

```
1980s: RPC (Remote Procedure Call)
  ↓
1998: XML-RPC
  ↓
2000: SOAP (Microsoft, IBM)
  ↓
2000: REST (Roy Fielding dissertation)
  ↓
2010s: RESTful API best practices
  ↓
2011: OpenAPI (Swagger)
  ↓
2015: GraphQL (Facebook)
  ↓
2016: gRPC (Google)
  ↓
2024: Teraz - wszystkie współistnieją!
```

**Kluczowa obserwacja:**
- SOAP = **standard** (szczegółowa specyfikacja)
- REST = **filozofia** (ogólne zasady)
- OpenAPI = **specyfikacja dla RESTful API** (dokumentacja)

---

## 3. SOAP (Simple Object Access Protocol)

**Lata 2000-2010 (dominacja), nadal używane w bankowości/enterprise**

### Czym jest SOAP?

**SOAP = pełny STANDARD komunikacji przez XML**

```xml
<!-- SOAP Request - pobierz użytkownika -->
POST /api HTTP/1.1
Content-Type: text/xml

<?xml version="1.0"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Header>
    <auth>Bearer token123</auth>
  </soap:Header>
  <soap:Body>
    <GetUser>
      <UserId>123</UserId>
    </GetUser>
  </soap:Body>
</soap:Envelope>
```

```xml
<!-- SOAP Response -->
<?xml version="1.0"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetUserResponse>
      <User>
        <Id>123</Id>
        <Name>John Doe</Name>
        <Email>john@example.com</Email>
      </User>
    </GetUserResponse>
  </soap:Body>
</soap:Envelope>
```

---

### WSDL - Samodokumentacja w SOAP

**WSDL (Web Services Description Language) = XML opisujący co API umie**

**Analogia:** WSDL to menu restauracji napisane w XML

```xml
<!-- users.wsdl - opisuje wszystkie operacje -->
<definitions name="UserService">
  <types>
    <!-- Definicje typów danych -->
    <schema>
      <element name="User">
        <complexType>
          <sequence>
            <element name="Id" type="int"/>
            <element name="Name" type="string"/>
            <element name="Email" type="string"/>
          </sequence>
        </complexType>
      </element>
    </schema>
  </types>

  <message name="GetUserRequest">
    <part name="UserId" type="int"/>
  </message>

  <message name="GetUserResponse">
    <part name="User" element="User"/>
  </message>

  <portType name="UserPortType">
    <operation name="GetUser">
      <input message="GetUserRequest"/>
      <output message="GetUserResponse"/>
    </operation>
  </portType>

  <binding name="UserBinding" type="UserPortType">
    <!-- HTTP/SOAP binding details -->
  </binding>

  <service name="UserService">
    <port name="UserPort" binding="UserBinding">
      <soap:address location="http://example.com/api/users"/>
    </port>
  </service>
</definitions>
```

**Co daje WSDL?**

1. **Auto-generowanie kodu klienta:**
   ```bash
   # Narzędzie czyta WSDL i generuje kod!
   wsdl2java users.wsdl
   # Tworzy klasy: UserService, User, GetUserRequest, etc.
   ```

2. **Silna typizacja:**
   - WSDL definiuje typy (int, string, complex types)
   - Kompilator sprawdzi błędy PRZED uruchomieniem

3. **Samodokumentacja:**
   - WSDL to dokumentacja API
   - Nie może być nieaktualna (to część API!)

---

### SOAP - Zalety i Wady

#### ✅ Zalety:

1. **Pełny standard** - wszystko zdefiniowane (security, transactions, error handling)
2. **WSDL** - auto-generowanie kodu, silna typizacja
3. **WS-Security** - built-in encryption, signing
4. **Transactions** - ACID transactions przez WS-AtomicTransaction
5. **Protocol-agnostic** - może działać na HTTP, SMTP, TCP

#### ❌ Wady:

1. **Verbose** - ogromne XML (100 linii dla prostej operacji)
2. **Wolny** - parsowanie XML jest kosztowne
3. **Skomplikowany** - WS-* standards (WS-Security, WS-Addressing, etc.)
4. **Trudny debugging** - XML jest trudny do czytania
5. **Overkill** - większość aplikacji nie potrzebuje takiej złożoności

---

### Przykład użycia SOAP (Python):

```python
from zeep import Client

# Klient automatycznie czyta WSDL i generuje metody!
client = Client('http://example.com/users.wsdl')

# Wywołanie operacji (auto-completion działa!)
user = client.service.GetUser(UserId=123)

print(user.Name)  # John Doe
print(user.Email)  # john@example.com
```

**WSDL = kontrakt między klientem a serwerem**

---

### Kiedy używać SOAP?

✅ **Użyj SOAP gdy:**
- Enterprise/bankowość (legacy systems)
- Potrzebujesz ACID transactions
- Potrzebujesz WS-Security (signing, encryption)
- Silna typizacja jest krytyczna

❌ **NIE używaj SOAP gdy:**
- Budujesz nowoczesne web/mobile API
- Potrzebujesz szybkości/prostoty
- JSON jest wystarczający

---

## 4. REST (Representational State Transfer)

**2000 - Roy Fielding (doktorat), ale popularne od ~2010**

### Czym jest REST?

**REST ≠ Standard!**

**REST = filozofia / architectural style / zestaw zasad**

**Analogia:** REST to jak zen - ogólne zasady, nie szczegółowa instrukcja

---

### 6 zasad REST (constraints):

#### 1. **Client-Server**

```
Klient (UI)  ←→  Serwer (Data/Logic)

- Separacja concerns
- Klient nie wie jak serwer przechowuje dane
- Serwer nie wie jak klient renderuje UI
```

#### 2. **Stateless**

```
Request 1: GET /users/123
Request 2: GET /users/456

- Każdy request jest niezależny
- Serwer NIE PAMIĘTA poprzednich requestów
- Wszystkie info w requeście (token, parametry)
```

**Przykład ZŁAMANY stateless:**
```python
# ❌ ŹLE - serwer pamięta stan sesji
GET /login → serwer zapisuje: user_id=123 w sesji
GET /profile → serwer czyta user_id z sesji

# ✅ DOBRZE - stateless
GET /login → zwraca token
GET /profile (Authorization: Bearer token) → token zawiera user_id
```

#### 3. **Cacheable**

```http
HTTP/1.1 200 OK
Cache-Control: max-age=3600  ← Response może być cache'owany!
ETag: "abc123"

{"id": 1, "name": "John"}
```

**Response musi mówić czy można go cache'ować**

#### 4. **Uniform Interface** (najważniejsze!)

**4 pod-zasady:**

**a) Resource-based (zasoby, nie akcje)**

```python
# ✅ DOBRZE - zasoby
GET    /users       # lista userów
GET    /users/123   # konkretny user
POST   /users       # utwórz user
PUT    /users/123   # zaktualizuj user
DELETE /users/123   # usuń user

# ❌ ŹLE - akcje (RPC style, nie REST!)
POST /getUser?id=123
POST /createUser
POST /updateUser
POST /deleteUser
```

**b) Reprezentacja zasobów**

```
Ten sam zasób (/users/123) może mieć różne reprezentacje:

GET /users/123
Accept: application/json  → {"id": 123, "name": "John"}

GET /users/123
Accept: application/xml   → <user><id>123</id><name>John</name></user>

GET /users/123
Accept: text/html         → <html><h1>John</h1></html>
```

**c) Self-descriptive messages**

```http
GET /users/123 HTTP/1.1
Accept: application/json

HTTP/1.1 200 OK
Content-Type: application/json  ← Header mówi co to jest!

{"id": 123, "name": "John"}
```

**Message zawiera wszystko co potrzebne do interpretacji**

**d) HATEOAS** (omówione osobno)

#### 5. **Layered System**

```
Klient → Load Balancer → Cache → API Gateway → Web Server → Database

- Klient nie wie o warstwach pośrednich
- Można dodać cache/load balancer bez zmiany klienta
```

#### 6. **Code on Demand** (opcjonalne)

```
Serwer może wysłać kod do wykonania na kliencie (JavaScript)

GET /users/123
→ Response zawiera JavaScript do wykonania

(Rzadko używane, opcjonalne)
```

---

### REST - Kluczowa obserwacja:

**REST NIE OKREŚLA:**
- Jakiego formatu używać (JSON? XML? YAML?)
- Jak nazywać endpointy (`/users` vs `/api/users`?)
- Jak zwracać błędy (status codes? error format?)
- Jak robić paginację
- Jak robić filtering/sorting

**REST to ZASADY, nie szczegółowa specyfikacja!**

Dlatego powstało **RESTful API** - best practices!

---

## 5. RESTful API - Best Practices

**RESTful = REST + konkretne praktyki (community consensus)**

### Dlaczego powstało?

**Problem:** REST to zasady, ale każdy implementował inaczej!

```python
# Wszyscy mówią "REST API", ale każdy robi inaczej:

# API 1:
GET /users?id=123

# API 2:
GET /api/v1/users/123

# API 3:
GET /user/123

# Wszystkie "RESTful", ale różne!
```

**RESTful API = ustandaryzowane praktyki społeczności**

---

### RESTful Best Practices:

#### 1. **Używaj rzeczowników (nie czasowników) w URL**

```python
# ✅ DOBRZE - zasoby (rzeczowniki)
GET    /users
GET    /users/123
POST   /users
PUT    /users/123
DELETE /users/123

GET    /users/123/posts       # posty użytkownika
GET    /users/123/posts/456   # konkretny post

# ❌ ŹLE - akcje (czasowniki)
GET  /getUsers
POST /createUser
POST /updateUser
POST /deleteUser
GET  /getUserPosts
```

#### 2. **Używaj HTTP methods**

| Method | Operacja | Idempotent? | Safe? |
|--------|----------|-------------|-------|
| GET | Pobierz zasób | ✅ TAK | ✅ TAK |
| POST | Utwórz zasób | ❌ NIE | ❌ NIE |
| PUT | Zaktualizuj (zastąp) | ✅ TAK | ❌ NIE |
| PATCH | Zaktualizuj (częściowo) | ❌ NIE | ❌ NIE |
| DELETE | Usuń zasób | ✅ TAK | ❌ NIE |

**Idempotent** = wywołanie N razy daje ten sam efekt co 1 raz
**Safe** = nie modyfikuje zasobu

```python
# GET - idempotent, safe
GET /users/123  # 10x → zawsze ten sam user, nie zmienia danych

# POST - NIE idempotent
POST /users (body: {"name": "John"})
# 3x → tworzy 3 userów! (user1, user2, user3)

# PUT - idempotent
PUT /users/123 (body: {"name": "John"})
# 10x → user 123 ma name="John" (ten sam efekt)

# DELETE - idempotent
DELETE /users/123
# 10x → user 123 jest usunięty (ten sam efekt)
```

#### 3. **Używaj HTTP status codes**

```python
# ✅ DOBRZE - właściwe status codes
GET /users/123
→ 200 OK (znaleziono)
→ 404 Not Found (nie ma)

POST /users
→ 201 Created (utworzono)
→ 400 Bad Request (błędne dane)
→ 409 Conflict (user już istnieje)

PUT /users/123
→ 200 OK (zaktualizowano)
→ 404 Not Found (nie ma)

DELETE /users/123
→ 204 No Content (usunięto)
→ 404 Not Found (już nie ma)

# ❌ ŹLE - wszystko zwraca 200
GET /users/999 → 200 OK {"error": "not found"}
POST /users (invalid data) → 200 OK {"error": "bad request"}
```

**Popularne status codes:**

```
2xx - Success
  200 OK - sukces
  201 Created - utworzono zasób
  204 No Content - sukces, brak body

4xx - Client Error
  400 Bad Request - błędne dane
  401 Unauthorized - brak auth
  403 Forbidden - brak uprawnień
  404 Not Found - nie znaleziono
  409 Conflict - konflikt (np. email istnieje)
  422 Unprocessable Entity - walidacja failed

5xx - Server Error
  500 Internal Server Error - błąd serwera
  503 Service Unavailable - serwis niedostępny
```

#### 4. **Wersjonowanie API**

```python
# Opcja 1: URL versioning (najpopularniejsze)
GET /api/v1/users
GET /api/v2/users  # Breaking changes

# Opcja 2: Header versioning
GET /api/users
Accept: application/vnd.myapi.v1+json

# Opcja 3: Query parameter (nie polecane)
GET /api/users?version=1
```

#### 5. **Filtering, Sorting, Pagination**

```python
# Filtering
GET /users?status=active
GET /users?age_min=18&age_max=65

# Sorting
GET /users?sort=name
GET /users?sort=-created_at  # Descending (minus)

# Pagination
GET /users?page=1&limit=20
# lub
GET /users?offset=0&limit=20

# Response:
{
  "data": [...],
  "pagination": {
    "total": 1000,
    "page": 1,
    "limit": 20,
    "total_pages": 50
  }
}
```

#### 6. **Consistent Error Format**

```python
# ✅ DOBRZE - spójny format błędów
HTTP/1.1 400 Bad Request
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "Invalid email format",
    "details": {
      "field": "email",
      "value": "not-an-email"
    }
  }
}

# ❌ ŹLE - niespójne błędy
# Endpoint 1:
{"error": "Invalid email"}

# Endpoint 2:
{"message": "Bad request", "code": 400}

# Endpoint 3:
{"errors": [{"msg": "Validation failed"}]}
```

#### 7. **Use JSON (not XML)**

```python
# ✅ DOBRZE - JSON (standard dla RESTful)
Content-Type: application/json
{"id": 123, "name": "John"}

# ❌ ŹLE - XML (verbose, trudny do parsowania)
Content-Type: application/xml
<user><id>123</id><name>John</name></user>
```

#### 8. **Nested Resources**

```python
# Posty użytkownika
GET /users/123/posts
GET /users/123/posts/456
POST /users/123/posts
DELETE /users/123/posts/456

# Komentarze do posta
GET /users/123/posts/456/comments
GET /users/123/posts/456/comments/789

# Ale NIE za głęboko! (max 2-3 poziomy)
# ❌ ŹLE - za głębokie zagnieżdżenie
GET /users/123/posts/456/comments/789/replies/999/likes

# ✅ LEPIEJ - flatten
GET /comments/789/replies
GET /replies/999/likes
```

---

### REST vs RESTful - Różnice:

| | REST | RESTful |
|---|------|----------|
| **Typ** | Filozofia / zasady | Konkretne praktyki |
| **Autor** | Roy Fielding (2000) | Community consensus |
| **Szczegółowość** | Ogólne constraints | Szczegółowe guidelines |
| **Format** | Nie określa | JSON (de facto standard) |
| **URL** | Resource-based | Konkretne konwencje (`/users`, nie `/getUsers`) |
| **Errors** | Nie określa | HTTP status codes + JSON format |
| **Versioning** | Nie określa | `/api/v1/` |

**RESTful = REST + community best practices**

---

## 6. HATEOAS (Hypermedia as the Engine of Application State)

**Ostatnia zasada REST - najmniej zrozumiana, rzadko implementowana**

### Czym jest HATEOAS?

**HATEOAS = API zwraca LINKI do powiązanych zasobów**

**Analogia:** Strona internetowa

```
Gdy odwiedzasz stronę Google.com:
- Nie musisz znać wszystkich URL
- Strona zawiera linki: <a href="/search">Search</a>
- Klikasz link → przechodzisz dalej
- NAWIGACJA przez linki, nie przez zapamiętywanie URL!

HATEOAS = to samo dla API!
```

---

### Przykład BEZ HATEOAS:

```python
# Request
GET /users/123

# Response (bez linków)
{
  "id": 123,
  "name": "John Doe",
  "email": "john@example.com"
}
```

**Problem:**
- Klient musi WIEDZIEĆ jak zbudować URL do postów: `/users/123/posts`
- Klient musi WIEDZIEĆ jak aktualizować usera: `PUT /users/123`
- Jeśli zmienisz URL → klient się psuje!

---

### Przykład Z HATEOAS:

```python
# Request
GET /users/123

# Response (z linkami!)
{
  "id": 123,
  "name": "John Doe",
  "email": "john@example.com",
  "_links": {
    "self": {
      "href": "/users/123"
    },
    "posts": {
      "href": "/users/123/posts"
    },
    "update": {
      "href": "/users/123",
      "method": "PUT"
    },
    "delete": {
      "href": "/users/123",
      "method": "DELETE"
    },
    "friends": {
      "href": "/users/123/friends"
    }
  }
}
```

**Korzyści:**
1. **Klient nie musi znać URL** - dostaje je w response!
2. **Zmiana URL** - zmienisz tylko serwer, klient działa dalej (używa linków)
3. **Discoverable API** - klient widzi co można zrobić

---

### HATEOAS - Nawigacja jak w przeglądarce:

```python
# Krok 1: Wejdź na API root
GET /api
{
  "_links": {
    "users": {"href": "/users"},
    "posts": {"href": "/posts"},
    "products": {"href": "/products"}
  }
}

# Krok 2: Kliknij link "users"
GET /users
{
  "data": [
    {
      "id": 123,
      "name": "John",
      "_links": {
        "self": {"href": "/users/123"}
      }
    }
  ],
  "_links": {
    "next": {"href": "/users?page=2"},
    "create": {"href": "/users", "method": "POST"}
  }
}

# Krok 3: Kliknij link "self" dla usera
GET /users/123
{
  "id": 123,
  "name": "John",
  "_links": {
    "posts": {"href": "/users/123/posts"},
    "friends": {"href": "/users/123/friends"}
  }
}

# Krok 4: Kliknij link "posts"
GET /users/123/posts
# ...
```

**Klient nawiguje przez API jak przez stronę WWW!**

---

### HAL (Hypertext Application Language) - Standard dla HATEOAS

**HAL = format JSON z linkami**

```python
# HAL format
{
  "_links": {
    "self": {"href": "/users/123"},
    "posts": {"href": "/users/123/posts"}
  },
  "id": 123,
  "name": "John Doe",
  "email": "john@example.com",
  "_embedded": {
    "latest_post": {
      "_links": {
        "self": {"href": "/posts/456"}
      },
      "id": 456,
      "title": "My latest post"
    }
  }
}
```

**HAL features:**
- `_links` - linki do powiązanych zasobów
- `_embedded` - zagnieżdżone zasoby (zamiast oddzielnych requestów)
- `self` - link do samego siebie

---

### HATEOAS - Dlaczego rzadko używane?

#### ❌ Wady:

1. **Verbose** - więcej danych w response
2. **Overkill** - większość klientów i tak ma hardcoded URL
3. **Trudne dla mobile** - mobile apps mają固定 UI, nie dynamiczną nawigację
4. **Brak wsparcia** - frameworki słabo wspierają HATEOAS

#### ✅ Zalety:

1. **Evolvable API** - zmieniasz URL bez łamania klientów
2. **Discoverable** - klient widzi możliwości
3. **Decoupling** - klient nie zna struktury URL

---

### Kiedy używać HATEOAS?

✅ **Użyj HATEOAS gdy:**
- Public API dla zewnętrznych devów
- API będzie często zmieniać URL
- Chcesz aby API było discoverable

❌ **NIE używaj gdy:**
- Internal API (twój frontend + backend)
- Mobile app (固定 UI)
- Prostota > flexibility

**Większość RESTful API NIE używa HATEOAS** (i to OK!)

---

## 7. OpenAPI (Swagger) - Specyfikacja dla RESTful API

**2011 - Swagger → 2016 - OpenAPI (przekazany do Linux Foundation)**

### Czym jest OpenAPI?

**OpenAPI = WSDL dla RESTful API**

**OpenAPI Specification (OAS) = plik YAML/JSON opisujący API**

**Analogia:**
- WSDL (SOAP) = menu restauracji w XML
- OpenAPI (REST) = menu restauracji w YAML

---

### Aktualna wersja: OpenAPI 3.1 (2021)

```
Historia:
- 2011: Swagger 1.0
- 2014: Swagger 2.0
- 2016: Swagger → OpenAPI 3.0 (przekazany do OAI)
- 2021: OpenAPI 3.1 (current)
```

**Swagger = narzędzia (Swagger UI, Swagger Editor)**
**OpenAPI = specyfikacja (standard)**

---

### Przykład OpenAPI 3.1:

```yaml
# openapi.yaml
openapi: 3.1.0
info:
  title: Users API
  version: 1.0.0
  description: API do zarządzania użytkownikami

servers:
  - url: https://api.example.com/v1
    description: Production server
  - url: http://localhost:8000
    description: Development server

paths:
  /users:
    get:
      summary: Pobierz listę użytkowników
      parameters:
        - name: page
          in: query
          schema:
            type: integer
            default: 1
        - name: limit
          in: query
          schema:
            type: integer
            default: 20
      responses:
        '200':
          description: Lista użytkowników
          content:
            application/json:
              schema:
                type: object
                properties:
                  data:
                    type: array
                    items:
                      $ref: '#/components/schemas/User'
                  pagination:
                    type: object
                    properties:
                      total:
                        type: integer
                      page:
                        type: integer
                      limit:
                        type: integer

    post:
      summary: Utwórz nowego użytkownika
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/UserCreate'
      responses:
        '201':
          description: Utworzono użytkownika
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/User'
        '400':
          description: Błędne dane
        '409':
          description: Email już istnieje

  /users/{userId}:
    get:
      summary: Pobierz użytkownika
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: integer
      responses:
        '200':
          description: Użytkownik znaleziony
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/User'
        '404':
          description: Nie znaleziono

    put:
      summary: Zaktualizuj użytkownika
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: integer
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/UserUpdate'
      responses:
        '200':
          description: Zaktualizowano
        '404':
          description: Nie znaleziono

    delete:
      summary: Usuń użytkownika
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: integer
      responses:
        '204':
          description: Usunięto
        '404':
          description: Nie znaleziono

components:
  schemas:
    User:
      type: object
      properties:
        id:
          type: integer
          example: 123
        name:
          type: string
          example: "John Doe"
        email:
          type: string
          format: email
          example: "john@example.com"
        created_at:
          type: string
          format: date-time

    UserCreate:
      type: object
      required:
        - name
        - email
      properties:
        name:
          type: string
          minLength: 2
        email:
          type: string
          format: email
        password:
          type: string
          minLength: 8

    UserUpdate:
      type: object
      properties:
        name:
          type: string
        email:
          type: string
          format: email

  securitySchemes:
    BearerAuth:
      type: http
      scheme: bearer
      bearerFormat: JWT

security:
  - BearerAuth: []
```

---

### Co daje OpenAPI?

#### 1. **Dokumentacja (Swagger UI)**

```bash
# Swagger UI generuje interaktywną dokumentację!
# http://localhost:8000/docs
```

**Swagger UI:**
- Czytasz OpenAPI spec → generuje piękną dokumentację
- **Try it out** - testujesz API z przeglądarki!
- Widzisz wszystkie endpointy, parametry, responses

#### 2. **Auto-generowanie kodu klienta**

```bash
# OpenAPI Generator (jak wsdl2java dla SOAP)
openapi-generator-cli generate \
  -i openapi.yaml \
  -g python \
  -o ./client

# Generuje:
# - ApiClient
# - UsersApi (z metodami: get_users, create_user, etc.)
# - User, UserCreate models
```

```python
# Wygenerowany kod klienta:
from client import UsersApi, ApiClient, Configuration

config = Configuration(host="http://localhost:8000")
client = ApiClient(config)
api = UsersApi(client)

# Auto-completion działa!
users = api.get_users(page=1, limit=20)
user = api.get_user(user_id=123)
```

#### 3. **Walidacja requestów/responses**

```python
# FastAPI automatycznie waliduje według OpenAPI!
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()  # Auto-generuje OpenAPI spec!

class UserCreate(BaseModel):
    name: str
    email: str

@app.post("/users")
def create_user(user: UserCreate):
    # FastAPI sprawdzi czy request pasuje do UserCreate!
    return {"id": 123, **user.dict()}

# OpenAPI spec: http://localhost:8000/openapi.json
# Swagger UI: http://localhost:8000/docs
```

#### 4. **API Testing**

```bash
# Narzędzia mogą testować API według OpenAPI spec
# Dredd, Schemathesis, etc.

schemathesis run openapi.yaml --base-url http://localhost:8000
# Auto-generuje testy dla każdego endpointu!
```

#### 5. **Contract Testing**

```
Backend Team       OpenAPI Spec       Frontend Team
     ↓                  ↓                    ↓
Implementuje      Kontrakt (spec)      Mock server
     ↓                  ↓                    ↓
Test: impl vs spec                  Test: UI vs mock
```

**Contract = OpenAPI spec**

---

### OpenAPI vs WSDL:

| | WSDL (SOAP) | OpenAPI (REST) |
|---|-------------|----------------|
| **Format** | XML | YAML/JSON |
| **Readable** | ❌ Trudny | ✅ Łatwy |
| **Auto-gen code** | ✅ TAK | ✅ TAK |
| **Documentation** | ❌ Brak UI | ✅ Swagger UI |
| **Validation** | ✅ Built-in | ✅ Przez frameworki |
| **Ecosystem** | ❌ Stary | ✅ Nowoczesny |

**OpenAPI = nowoczesna alternatywa dla WSDL**

---

### OpenAPI 3.0 vs 3.1 - Różnice:

**OpenAPI 3.1 (2021) - aktualna:**
- ✅ Zgodność z JSON Schema 2020-12
- ✅ `webhooks` support
- ✅ Lepsze `$ref` (circular refs)
- ✅ `const` keyword

**OpenAPI 3.0 (2017):**
- Nadal szeroko używana
- Większość narzędzi wspiera

**Rekomendacja: Używaj 3.1 dla nowych projektów**

---

### FastAPI - OpenAPI out of the box:

```python
from fastapi import FastAPI

app = FastAPI(
    title="Users API",
    version="1.0.0",
    description="API do zarządzania użytkownikami"
)

@app.get("/users")
def get_users():
    """Pobierz listę użytkowników"""
    return [{"id": 1, "name": "John"}]

# FastAPI automatycznie generuje:
# - OpenAPI spec: /openapi.json
# - Swagger UI: /docs
# - ReDoc: /redoc
```

**FastAPI = OpenAPI first!**

---

## 8. GraphQL - Alternatywa dla REST

**2015 - Facebook (open-sourced)**

### Czym jest GraphQL?

**GraphQL = query language dla API**

**Problem z REST:**

```python
# REST - potrzebujesz 3 requesty dla user + posts + comments
GET /users/123
GET /users/123/posts
GET /posts/456/comments

# Albo endpoint zwraca za dużo danych:
GET /users/123?include=posts,comments
# Response: 1MB danych (a potrzebujesz tylko name + email!)
```

**Over-fetching** = dostajesz za dużo danych
**Under-fetching** = musisz robić wiele requestów

---

### GraphQL - Rozwiązanie:

```graphql
# 1 request, dokładnie to czego potrzebujesz!
POST /graphql

{
  user(id: 123) {
    name
    email
    posts {
      title
      comments {
        text
        author {
          name
        }
      }
    }
  }
}
```

```json
// Response - dokładnie to czego żądałeś!
{
  "data": {
    "user": {
      "name": "John Doe",
      "email": "john@example.com",
      "posts": [
        {
          "title": "My post",
          "comments": [
            {
              "text": "Great post!",
              "author": {"name": "Jane"}
            }
          ]
        }
      ]
    }
  }
}
```

**Klient decyduje co dostanie!**

---

### GraphQL - Schema (jak WSDL/OpenAPI):

```graphql
# schema.graphql
type User {
  id: ID!
  name: String!
  email: String!
  posts: [Post!]!
}

type Post {
  id: ID!
  title: String!
  content: String!
  author: User!
  comments: [Comment!]!
}

type Comment {
  id: ID!
  text: String!
  author: User!
}

type Query {
  user(id: ID!): User
  users: [User!]!
  post(id: ID!): Post
}

type Mutation {
  createUser(name: String!, email: String!): User!
  updateUser(id: ID!, name: String, email: String): User!
  deleteUser(id: ID!): Boolean!
}
```

**Schema = kontrakt API (jak WSDL/OpenAPI)**

---

### GraphQL - Zalety i Wady:

#### ✅ Zalety:

1. **1 request zamiast N** - nie ma under-fetching
2. **Dokładnie to czego potrzebujesz** - nie ma over-fetching
3. **Silna typizacja** - schema definiuje typy
4. **Self-documenting** - GraphiQL/Apollo Playground (jak Swagger UI)
5. **Versionless** - dodajesz pola, nie łamiesz starych klientów

#### ❌ Wady:

1. **Skomplikowane** - trudniejsze niż REST
2. **Caching** - trudny (każdy query inny!)
3. **Rate limiting** - trudny (query może być dowolnie kosztowny)
4. **N+1 problem** - łatwo napisać nieefektywne queries
5. **Overkill** - dla prostych CRUD API REST wystarcza

---

### Kiedy używać GraphQL?

✅ **Użyj GraphQL gdy:**
- Mobile app (oszczędność bandwidth)
- Skomplikowane relacje między danymi
- Wielu klientów z różnymi potrzebami
- Potrzebujesz elastyczności (różne queries)

❌ **NIE używaj gdy:**
- Prosty CRUD API
- File uploads (REST/HTTP lepszy)
- Real-time streaming (WebSockets lepsze)
- Team nie zna GraphQL (learning curve)

---

## 9. gRPC - Google Remote Procedure Call

**2016 - Google (open-sourced)**

### Czym jest gRPC?

**gRPC = nowoczesny RPC (jak SOAP, ale szybszy)**

**gRPC = HTTP/2 + Protocol Buffers (binary format)**

---

### REST vs gRPC:

```
REST:
- HTTP/1.1
- JSON (text)
- Resource-based (/users/123)
- Human-readable

gRPC:
- HTTP/2
- Protocol Buffers (binary)
- Action-based (GetUser(123))
- Machine-only (binary)
```

---

### Protocol Buffers (Protobuf):

```protobuf
// users.proto
syntax = "proto3";

service UserService {
  rpc GetUser(GetUserRequest) returns (User);
  rpc ListUsers(ListUsersRequest) returns (ListUsersResponse);
  rpc CreateUser(CreateUserRequest) returns (User);
}

message User {
  int32 id = 1;
  string name = 2;
  string email = 3;
}

message GetUserRequest {
  int32 id = 1;
}

message ListUsersRequest {
  int32 page = 1;
  int32 limit = 2;
}

message ListUsersResponse {
  repeated User users = 1;
}

message CreateUserRequest {
  string name = 1;
  string email = 2;
}
```

**Protobuf = schema (jak WSDL/OpenAPI/GraphQL schema)**

---

### Auto-generowanie kodu:

```bash
# Kompilacja .proto → kod Pythona/Java/Go/etc.
protoc --python_out=. users.proto

# Generuje:
# - users_pb2.py (messages: User, GetUserRequest, etc.)
# - users_pb2_grpc.py (service: UserServiceStub)
```

```python
# Client (auto-generated)
import grpc
import users_pb2
import users_pb2_grpc

channel = grpc.insecure_channel('localhost:50051')
stub = users_pb2_grpc.UserServiceStub(channel)

# Wywołanie RPC (jak funkcja!)
request = users_pb2.GetUserRequest(id=123)
user = stub.GetUser(request)

print(user.name)  # John Doe
```

**gRPC = wywołujesz funkcję na serwerze (jak lokalnie!)**

---

### gRPC - 4 typy RPC:

#### 1. **Unary RPC** (request-response, jak REST)

```protobuf
rpc GetUser(GetUserRequest) returns (User);
```

```
Client → Request → Server
Client ← Response ← Server
```

#### 2. **Server Streaming** (1 request → stream responses)

```protobuf
rpc ListUsers(ListUsersRequest) returns (stream User);
```

```
Client → Request → Server
Client ← User 1
Client ← User 2
Client ← User 3
...
```

#### 3. **Client Streaming** (stream requests → 1 response)

```protobuf
rpc UploadUsers(stream CreateUserRequest) returns (UploadSummary);
```

```
Client → User 1 → Server
Client → User 2
Client → User 3
...
Client ← Summary
```

#### 4. **Bidirectional Streaming** (stream obustronne)

```protobuf
rpc Chat(stream ChatMessage) returns (stream ChatMessage);
```

```
Client → Message 1 → Server
Client ← Message 2 ← Server
Client → Message 3 → Server
...
```

**gRPC = streaming out of the box! (REST nie ma tego)**

---

### gRPC - Zalety i Wady:

#### ✅ Zalety:

1. **Szybki** - binary (Protobuf) vs JSON (text)
   - 5-10x mniejszy payload
   - 5-10x szybszy parsing
2. **HTTP/2** - multiplexing, streaming, header compression
3. **Streaming** - 4 typy (unary, server, client, bidirectional)
4. **Silna typizacja** - Protobuf schema
5. **Multi-language** - auto-gen dla 10+ języków

#### ❌ Wady:

1. **Binary** - nie można czytać w przeglądarce
2. **Browser support** - słabe (gRPC-Web workaround)
3. **HTTP/2 required** - nie wszystkie proxy/load balancery wspierają
4. **Learning curve** - trudniejsze niż REST
5. **Debugging** - trudny (binary, nie text)

---

### Kiedy używać gRPC?

✅ **Użyj gRPC gdy:**
- Microservices communication (internal)
- Performance critical (latency/bandwidth)
- Streaming (real-time data)
- Polyglot environment (Java + Python + Go)

❌ **NIE używaj gdy:**
- Public API (REST lepszy)
- Browser-based (gRPC-Web nie jest natywny)
- Potrzebujesz human-readable (JSON)

---

## 10. Porównanie WSZYSTKICH

### Chronologia:

```
1980s  RPC (Remote Procedure Call)
1998   XML-RPC
2000   SOAP (pełny standard, WSDL)
2000   REST (filozofia, Roy Fielding)
2010s  RESTful API (community practices)
2011   OpenAPI/Swagger (spec dla RESTful)
2015   GraphQL (Facebook)
2016   gRPC (Google)
```

---

### Porównanie:

| | SOAP | REST | GraphQL | gRPC |
|---|------|------|---------|------|
| **Typ** | Standard | Filozofia | Query Language | RPC Framework |
| **Format** | XML | JSON | JSON | Binary (Protobuf) |
| **Protocol** | HTTP, SMTP, TCP | HTTP | HTTP | HTTP/2 |
| **Schema** | WSDL (XML) | OpenAPI (YAML) | GraphQL Schema | Protobuf (.proto) |
| **Readable** | ❌ XML verbose | ✅ JSON | ✅ JSON | ❌ Binary |
| **Performance** | ❌ Wolny | ⚠️ OK | ⚠️ OK | ✅ Szybki |
| **Streaming** | ❌ NIE | ❌ NIE | ⚠️ Subscriptions | ✅ TAK (4 typy) |
| **Browser** | ⚠️ Trudny | ✅ Natywny | ✅ Natywny | ❌ gRPC-Web |
| **Typizacja** | ✅ Silna | ⚠️ Opcjonalna | ✅ Silna | ✅ Silna |
| **Caching** | ❌ Trudny | ✅ HTTP cache | ❌ Trudny | ❌ Brak |
| **Versioning** | ⚠️ Namespace | ✅ /v1/, /v2/ | ✅ Additive | ⚠️ Protobuf versioning |
| **Use Case** | Enterprise/Bank | Public API | Mobile/Complex | Microservices |

---

### Kiedy co używać?

#### SOAP:
```
✅ Enterprise/bankowość (legacy)
✅ ACID transactions wymagane
✅ WS-Security (signing, encryption)
❌ Nowoczesne web/mobile API
```

#### RESTful API:
```
✅ Public API (developer-friendly)
✅ Web/mobile apps
✅ CRUD operations
✅ Caching ważny
✅ Default choice (prostota + popularność)
```

#### GraphQL:
```
✅ Mobile apps (oszczędność bandwidth)
✅ Complex nested data
✅ Wielu klientów (różne potrzeby)
✅ Frontend-driven API
❌ Prosty CRUD
❌ File uploads
```

#### gRPC:
```
✅ Microservices (internal communication)
✅ Performance critical
✅ Streaming (real-time)
✅ Polyglot services (Java + Python + Go)
❌ Public API (not browser-friendly)
❌ Human-readable wymagane
```

---

### Przykład architektury (Netflix-style):

```
Mobile App ────── GraphQL ───────┐
                                 |
Web App ──────── REST API ───────┼──→ API Gateway
                                 |        ↓
Admin Panel ──── REST API ───────┘    (konwertuje)
                                         ↓
                            ┌────────────┼────────────┐
                            ↓            ↓            ↓
                       User Service  Order Service  Payment Service
                            ↑            ↑            ↑
                            └─── gRPC ───┴─── gRPC ───┘
                         (internal microservices communication)
```

**Używaj różnych stylów na różnych warstwach!**

---

## 11. Inne ważne koncepty

### WebSockets (Real-time communication)

**Problem:**
- REST = request-response (klient inicjuje)
- Jak serwer może wysłać dane do klienta?

**Rozwiązania:**

#### 1. **Polling** (REST)
```python
# Klient pyta co 5 sekund: "Są nowe wiadomości?"
setInterval(() => {
  fetch('/api/messages')  # Request co 5s
}, 5000)

# ❌ Nieefektywne (99% requestów = "brak zmian")
```

#### 2. **Long Polling** (REST)
```python
# Klient: "Poczekaj aż będą dane"
GET /api/messages/wait
# Server czeka (30s) aż będą dane → Response
# Klient od razu wysyła nowy request

# ⚠️ Lepsze, ale nadal suboptymalne
```

#### 3. **WebSockets** (Full-duplex)
```python
# Klient: Otwórz połączenie WebSocket
ws = new WebSocket('ws://example.com/chat')

# Bidirectional communication!
Client → Server: "Hello"
Server → Client: "Welcome!"  # Server może wysłać pierwszy!
Server → Client: "New message from John"
Client → Server: "Thanks"

# ✅ Real-time, efektywne
```

**Use cases:**
- Chat apps
- Live notifications
- Collaborative editing (Google Docs)
- Live dashboards

---

### Server-Sent Events (SSE)

```python
# Server → Client (one-way)
GET /api/events
Accept: text/event-stream

# Server wysyła events:
data: {"type": "new_message", "text": "Hello"}

data: {"type": "user_joined", "user": "John"}

# ✅ Prostsze niż WebSockets (tylko Server → Client)
```

**Use cases:**
- Live updates (stock prices)
- Notifications
- Progress tracking

---

### Webhooks

**Webhook = "odwrotne API" (serwer wywołuje klienta)**

```python
# Setup webhook:
POST /api/webhooks
{
  "url": "https://your-app.com/webhook",
  "events": ["payment.success", "payment.failed"]
}

# Gdy event występuje, serwer wywołuje Twój URL:
POST https://your-app.com/webhook
{
  "event": "payment.success",
  "payment_id": 123,
  "amount": 100
}

# Twoja aplikacja dostaje powiadomienie!
```

**Use cases:**
- Payment notifications (Stripe)
- GitHub push events
- CI/CD triggers

---

### JSON:API - Standard formatowania

**Problem:** RESTful API ma różne formaty

**JSON:API = specyfikacja formatu JSON dla REST**

```json
{
  "data": {
    "type": "users",
    "id": "123",
    "attributes": {
      "name": "John Doe",
      "email": "john@example.com"
    },
    "relationships": {
      "posts": {
        "links": {
          "related": "/users/123/posts"
        }
      }
    }
  }
}
```

**JSON:API = konwencje dla REST (jak HAL dla HATEOAS)**

---

## 🎯 Podsumowanie

### Timeline:

```
2000: SOAP (standard, WSDL, XML) - enterprise
  ↓
2000: REST (filozofia) - proste, resource-based
  ↓
2010s: RESTful (best practices) - JSON, HTTP status codes
  ↓
2011: OpenAPI (spec dla RESTful) - dokumentacja, code gen
  ↓
2015: GraphQL (query language) - flexible, mobile-friendly
  ↓
2016: gRPC (RPC framework) - fast, streaming, microservices
```

---

### Kluczowe rozróżnienia:

#### 1. **Standard vs Filozofia:**
- **SOAP = STANDARD** (szczegółowa specyfikacja)
- **REST = FILOZOFIA** (ogólne zasady)
- **RESTful = PRAKTYKI** (community consensus)
- **OpenAPI = SPECYFIKACJA** (dla RESTful)

#### 2. **Samodokumentacja:**
- **SOAP:** WSDL (XML) - auto-gen kodu
- **REST:** Brak (ale OpenAPI!)
- **OpenAPI:** YAML/JSON - Swagger UI, auto-gen
- **GraphQL:** Schema - GraphiQL, auto-gen
- **gRPC:** Protobuf - auto-gen dla 10+ języków
- **HATEOAS:** Linki w response (jak HTML)

#### 3. **Format:**
- SOAP → XML (verbose)
- REST → JSON (default)
- GraphQL → JSON
- gRPC → Binary (Protobuf)

---

### Default choice w 2024:

```python
# Public API / Web / Mobile:
RESTful API + OpenAPI (FastAPI)

# Microservices (internal):
gRPC

# Complex data / Mobile-first:
GraphQL

# Enterprise / Legacy:
SOAP (tylko jeśli musisz)
```

---

### Co zapamiętać:

1. **SOAP** = pełny standard, WSDL (XML), enterprise, wolny
2. **REST** = filozofia (6 zasad), nie standard!
3. **RESTful** = REST + community practices (JSON, status codes, versioning)
4. **HATEOAS** = linki w response (rzadko używane)
5. **OpenAPI** = WSDL dla RESTful (YAML, Swagger UI, auto-gen)
6. **GraphQL** = query language (1 request, dokładnie czego potrzebujesz)
7. **gRPC** = RPC (binary, HTTP/2, streaming, microservices)

**Najważniejsze:** Używaj odpowiedniego narzędzia do zadania!

---